# Face Anonymization - Sticker Detection Debug

This notebook demonstrates the current state of sticker detection for photogrammetry scans.

In [1]:
import numpy as np
import pyvista as pv

import cedalion
import cedalion.io
import cedalion.plots
from cedalion import units
from cedalion.geometry.photogrammetry.processors import ColoredStickerProcessor

pv.set_jupyter_backend("server")

## 1. Load and View Scan

In [4]:
SUBJECT_NUMBER = 11
SCANS_FOLDER = "/home/ma7/BA/PG_Subjects11-15"

scan_path = f"{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}.obj"
surface = cedalion.io.read_einstar_obj(scan_path)

print(f"Loaded: {surface.nvertices:,} vertices, {surface.nfaces:,} faces")

# Visualize the scan
pvplt = pv.Plotter()
cedalion.plots.plot_surface(pvplt, surface, opacity=1.0)
pvplt.add_text(f"Subject {SUBJECT_NUMBER} - Original Scan", position="upper_left", font_size=14)
pvplt.show()

Loaded: 450,295 vertices, 848,480 faces


Widget(value='<iframe src="http://localhost:42449/index.html?ui=P_0x7f02bc9f7e10_2&reconnect=auto" class="pyvi…

## 2. Sticker Detection

We need to detect 5 colored stickers placed at landmark positions (Nz, Iz, Cz, LPA, RPA).

In [11]:
# Sticker detection with current parameters
# Colors: (hue_min, hue_max, value_min, value_max)
STICKER_COLORS = {
    "G": (0.20, 0.45, 0.5, 1.0),   # Green
    "B": (0.50, 0.75, 0.5, 1.0),   # Blue  
}

processor = ColoredStickerProcessor(
    colors=STICKER_COLORS,
    sticker_radius=6.5 * units.mm,
    saturation_min=0.6,
    min_nvertices=50,
)

sticker_positions, sticker_normals = processor.process(surface)
positions = sticker_positions.pint.dequantify().values

print(f"Detected {len(positions)} stickers (expected: 5)")
print(f"\nSticker positions:")
for i, pos in enumerate(positions):
    print(f"  {i+1}: X={pos[0]:7.1f}, Y={pos[1]:7.1f}, Z={pos[2]:7.1f}")

[[-43.676117 -26.677429 691.970337]
 [148.323853 107.722595 443.313904]
 [193.123901  82.922577 443.702026]
 ...
 [150.723877 106.122559 443.925629]
 [135.858337 113.587158 443.246857]
 [135.311035 113.214905 443.769348]]
[[-102.876129  152.522583  520.006531]
 [-103.676117  152.062683  519.958191]
 [ 162.220581   -7.773804  486.627808]
 [ 163.123901   -6.277435  486.89386 ]
 [ 162.723877   -6.677429  486.865021]
 [ 162.323853   -7.077423  486.891724]
 [ 161.923889   -7.077423  487.043976]
 [ 161.923889   -7.477448  486.849548]
 [ 161.523865   -7.877441  486.886047]
 [ 161.123901   -8.277435  486.878723]
 [ 164.323853   -6.277435  486.448547]
 [ 163.923889   -6.277435  486.554749]
 [ 163.523865   -6.277435  486.670349]
 [ 162.723877   -7.077423  486.694794]
 [ 162.723877   -7.477448  486.558472]
 [ 162.323853   -7.477448  486.696259]
 [ 164.723877   -5.077423  486.757751]
 [ 165.123901   -5.477448  486.393524]
 [ 164.723877   -5.477448  486.577332]
 [ 164.723877   -5.877441  486.391174

In [12]:
# Visualize detected stickers on the scan
pvplt = pv.Plotter()
cedalion.plots.plot_surface(pvplt, surface, opacity=0.7)

if len(positions) > 0:
    sticker_cloud = pv.PolyData(positions)
    pvplt.add_mesh(sticker_cloud, color="red", point_size=20, render_points_as_spheres=True)

pvplt.add_text(f"Detected {len(positions)} stickers (expected: 5)", position="upper_left", font_size=14)
pvplt.show()

Widget(value='<iframe src="http://localhost:46441/index.html?ui=P_0x7f12d4592450_2&reconnect=auto" class="pyvi…

## 3. Debug: Analyze Colored Vertices

Let's examine the HSV distribution of colored vertices to understand why detection is failing.

In [ ]:
import colorsys

vertex_colors = surface.mesh.visual.to_color().vertex_colors
vertices = surface.mesh.vertices

# Convert RGB to HSV
rgb_to_hsv = np.vectorize(colorsys.rgb_to_hsv)
h, s, v = rgb_to_hsv(vertex_colors[:, 0], vertex_colors[:, 1], vertex_colors[:, 2])
v = v / 255.0  # Normalize value to 0-1

# Find highly saturated vertices (likely stickers)
high_sat_mask = (s > 0.5) & (v > 0.3)
print(f"Highly saturated vertices (s>0.5, v>0.3): {high_sat_mask.sum():,}")

# Analyze by hue ranges
print("\nHue distribution of saturated vertices:")
print(f"  Green (0.20-0.45): {((h >= 0.20) & (h <= 0.45) & high_sat_mask).sum():,}")
print(f"  Blue (0.55-0.70):  {((h >= 0.55) & (h <= 0.70) & high_sat_mask).sum():,}")
print(f"  Red (0.0-0.08):    {((h >= 0.0) & (h <= 0.08) & high_sat_mask).sum():,}")

In [ ]:
# Visualize ALL colored vertices to see where the "stickers" actually are
green_mask = (h >= 0.20) & (h <= 0.45) & (s > 0.5) & (v > 0.3)
blue_mask = (h >= 0.55) & (h <= 0.70) & (s > 0.5) & (v > 0.3)

pvplt = pv.Plotter()
cedalion.plots.plot_surface(pvplt, surface, opacity=0.5)

# Show green vertices
if green_mask.sum() > 0:
    green_pts = pv.PolyData(vertices[green_mask])
    pvplt.add_mesh(green_pts, color="green", point_size=5, render_points_as_spheres=True)

# Show blue vertices  
if blue_mask.sum() > 0:
    blue_pts = pv.PolyData(vertices[blue_mask])
    pvplt.add_mesh(blue_pts, color="blue", point_size=5, render_points_as_spheres=True)

pvplt.add_text(f"Green: {green_mask.sum():,}  Blue: {blue_mask.sum():,}", position="upper_left", font_size=14)
pvplt.show()

## 4. Discussion Points for Professor

**Current Issue:** The sticker detection is not reliably finding exactly 5 stickers.

**Questions to discuss:**
1. What colors are the actual stickers on the scan? (Green/Blue/Red?)
2. Are there other colored objects in the scan causing false positives?
3. Should we use a different detection method (e.g., geometric shape instead of color)?